In [ ]:
# In[1]:

import pandas as pd

# Load CSV files
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Helper to build a one-row summary dataframe for files that have a 'kpi/trace/log name' column
def build_summary(df, file_label, name_col=None, top_kpi_limit=None):
    total_rows = len(df)
    min_ts = int(df['timestamp'].min()) if total_rows > 0 else pd.NA
    max_ts = int(df['timestamp'].max()) if total_rows > 0 else pd.NA
    top_cmdb = df['cmdb_id'].value_counts().head(10).reset_index()
    top_cmdb.columns = ['cmdb_id', 'count']
    # unique names limited to 50 if name_col provided
    if name_col and name_col in df.columns:
        unique_names = pd.Series(sorted(df[name_col].dropna().unique())[:50], name='unique_names')
    else:
        unique_names = pd.Series([], name='unique_names', dtype=object)
    summary = pd.DataFrame([{
        'file': file_label,
        'total_rows': total_rows,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts
    }])
    # optionally top kpi counts for metric
    top_kpis = None
    if name_col and top_kpi_limit:
        top_kpis = df[name_col].value_counts().head(top_kpi_limit).reset_index()
        top_kpis.columns = [name_col, 'count']
    return summary, top_cmdb, unique_names, top_kpis

# Build summaries
summary_metric, metric_top_cmdb, metric_unique_kpis, metric_top_kpis = build_summary(df_metric, "metric.csv", name_col='kpi_name', top_kpi_limit=20)
summary_trace, trace_top_cmdb, trace_unique_traces, _ = build_summary(df_trace, "trace.csv", name_col='trace_name')
summary_log, log_top_cmdb, log_unique_logs, _ = build_summary(df_log, "log.csv", name_col='log_name')
summary_error, error_top_cmdb, _, _ = build_summary(df_error, "error_logs.csv", name_col=None)

# Keep outputs compact. Display single-row summaries and top lists as small DataFrames/Series.
summary_metric, metric_top_cmdb.head(10), metric_unique_kpis.head(50), metric_top_kpis.head(20), summary_trace, trace_top_cmdb.head(10), trace_unique_traces.head(50), summary_log, log_top_cmdb.head(10), log_unique_logs.head(50), summary_error, error_top_cmdb.head(10)

```
Out[1]:
```
```python
# Build a plain-English summary string from previously computed summary variables
# Reuse existing variables: summary_metric, metric_top_cmdb, metric_unique_kpis, metric_top_kpis,
# summary_trace, trace_top_cmdb, trace_unique_traces, summary_log, log_top_cmdb, log_unique_logs,
# summary_error, error_top_cmdb

# Helpers to format top lists
def fmt_top_df(df, id_col='cmdb_id', count_col='count', limit=10):
    rows = []
    for _, r in df.head(limit).iterrows():
        rows.append(f"{r[id_col]} ({int(r[count_col])})")
    return ", ".join(rows) if rows else "none"

def fmt_name_list(series, limit=20):
    names = list(series.dropna().astype(str))[:limit]
    if not names:
        return "none"
    suffix = "" if len(names) <= limit else f", +{len(names)-limit} more"
    return ", ".join(names) + suffix

# Metric summary
m = summary_metric.iloc[0]
metric_top_list = fmt_top_df(metric_top_cmdb)
metric_unique_list = fmt_name_list(metric_unique_kpis, limit=50)
metric_top_kpis_list = ", ".join([f"{row.kpi_name} ({int(row['count'])})" for _, row in metric_top_kpis.reset_index(drop=True).head(20).iterrows()])

# Trace summary
t = summary_trace.iloc[0]
trace_top_list = fmt_top_df(trace_top_cmdb)
trace_unique_list = fmt_name_list(trace_unique_traces, limit=50)

# Log summary
l = summary_log.iloc[0]
log_top_list = fmt_top_df(log_top_cmdb)
log_unique_list = fmt_name_list(log_unique_logs, limit=50)

# Error logs
e = summary_error.iloc[0]
error_rows = int(e['total_rows']) if pd.notna(e['total_rows']) else 0

summary = (
    f"Summary of telemetry files (plain English):\n\n"
    f"metric.csv:\n"
    f"- Total rows: {int(m['total_rows'])}\n"
    f"- Min timestamp: {int(m['min_timestamp'])}, Max timestamp: {int(m['max_timestamp'])}\n"
    f"- Top cmdb_id values (up to 10): {metric_top_list}\n"
    f"- Unique kpi_name values (up to 50 shown): {metric_unique_list}\n"
    f"- Top 20 kpi_name counts: {metric_top_kpis_list}\n\n"
    f"trace.csv:\n"
    f"- Total rows: {int(t['total_rows'])}\n"
    f"- Min timestamp: {int(t['min_timestamp'])}, Max timestamp: {int(t['max_timestamp'])}\n"
    f"- Top cmdb_id values (up to 10): {trace_top_list}\n"
    f"- Unique trace_name values (up to 50 shown): {trace_unique_list}\n\n"
    f"log.csv:\n"
    f"- Total rows: {int(l['total_rows'])}\n"
    f"- Min timestamp: {int(l['min_timestamp'])}, Max timestamp: {int(l['max_timestamp'])}\n"
    f"- Top cmdb_id values (up to 10): {log_top_list}\n"
    f"- Unique log_name values (up to 50 shown): {log_unique_list}\n\n"
    f"error_logs.csv:\n"
    f"- Total rows: {error_rows}\n"
    f"- No timestamps or cmdb entries (empty)\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         file  total_rows  min_timestamp  max_timestamp
0  metric.csv        9129     1705982280     1705983720,                    cmdb_id  count
0  ts-admin-travel-service    200
1      ts-contacts-service    200
2          ts-auth-service    200
3     ts-assurance-service    200
4         ts-order-service    200
5          ts-ui-dashboard    200
6   ts-order-other-service    200
7          ts-seat-service    195
8       ts-travel2-service    195
9         ts-price-service    189, 0           cpu
1        diskio
2         error
3    latency-50
4    latency-90
5           mem
6        socket
7      workload
Name: unique_names, dtype: object,      kpi_name  count
0         cpu   1700
1         mem   1700
2      socket   1700
3      diskio   1675
4    workload    700
5  latency-50    699
6  latency-90    699
7       error    256,         file  total_rows  min_timestamp  max_timestamp
0  trace.csv       18272     1705982280     1705983720,                      cmdb_id  count
0                       root   1348
1        ts-preserve-service   1220
2          ts-travel-service   1188
3            ts-seat-service   1080
4         ts-travel2-service   1080
5  ts-preserve-other-service   1052
6           ts-order-service    876
7     ts-order-other-service    864
8         ts-station-service    768
9            ts-food-service    736, 0                         trace.from_root.duration_mean
1                          trace.from_root.duration_p95
2                            trace.from_root.error_rate
3                             trace.from_root.row_count
4     trace.from_ts-admin-basic-info-service.duratio...
                            ...                        
45              trace.from_ts-food-service.duration_p95
46                trace.from_ts-food-service.error_rate
47                 trace.from_ts-food-service.row_count
48    trace.from_ts-inside-payment-service.duration_...
49    trace.from_ts-inside-payment-service.duration_p95
Name: unique_names, Length: 50, dtype: object,       file  total_rows  min_timestamp  max_timestamp
0  log.csv        2178     1705982280     1705983720,                        cmdb_id  count
0  ts-admin-basic-info-service     50
1              ts-auth-service     50
2             ts-basic-service     50
3              ts-ui-dashboard     50
4       ts-order-other-service     50
5             ts-order-service     50
6             ts-price-service     50
7             ts-route-service     50
8              ts-seat-service     50
9            ts-travel-service     50, 0    log.error_count
1      log.row_count
Name: unique_names, dtype: object,              file  total_rows min_timestamp max_timestamp
0  error_logs.csv           0          <NA>          <NA>, Empty DataFrame
Columns: [cmdb_id, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure timestamp column is integer type
df_metric['timestamp'] = df_metric['timestamp'].astype(int)

# 1) Compute global P95 per (cmdb_id, kpi_name) using full series (no window filtering)
p95 = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)

# Count total distinct component-KPI series scanned
total_series_count = p95.shape[0]

# 2) Merge global_p95 back into full metric dataframe
df_metric_p = df_metric.merge(p95, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Parse timestamps to UTC datetimes (per rule 10) for filtering
df_metric_p['timestamp_dt'] = pd.to_datetime(df_metric_p['timestamp'], unit='s', utc=True)

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-23 03:55:03", utc=True)
window_end   = pd.to_datetime("2024-01-23 04:25:03", utc=True)

# 4) Filter to incident window (inclusive)
df_window = df_metric_p[(df_metric_p['timestamp_dt'] >= window_start) & (df_metric_p['timestamp_dt'] <= window_end)].copy()

# 5) Compute exceed boolean using global_p95 (computed from full series)
df_window['exceed'] = df_window['value'] >= df_window['global_p95']

# 6) Aggregations per (cmdb_id, kpi_name)
agg = (
    df_window
    .groupby(['cmdb_id', 'kpi_name', 'global_p95'], observed=True)
    .agg(
        window_total_points = ('value', 'size'),
        window_count_exceed_p95 = ('exceed', 'sum'),
        window_max_value = ('value', 'max')
    )
    .reset_index()
)

# 7) Earliest exceed timestamp per group (raw Unix timestamp). If none, will be missing.
earliest_exceed = (
    df_window[df_window['exceed']]
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp': 'window_earliest_exceed_timestamp'})
)

# 8) Merge earliest exceed into agg
result = agg.merge(earliest_exceed, on=['cmdb_id', 'kpi_name'], how='left')

# 9) Keep only rows where window_count_exceed_p95 > 0
result = result[result['window_count_exceed_p95'] > 0].copy()

# 10) Sort by window_count_exceed_p95 descending and limit to top 20
result = result.sort_values(by='window_count_exceed_p95', ascending=False).head(20)

# 11) Ensure columns order and compact display
result_df = result[
    ['cmdb_id', 'kpi_name', 'global_p95', 'window_total_points',
     'window_count_exceed_p95', 'window_earliest_exceed_timestamp', 'window_max_value']
].reset_index(drop=True)

# Display the compact results and the total number of distinct series scanned
result_df, total_series_count

```
Out[2]:
```
```python
# Build a plain-English summary string from the previously computed results
# Reuse existing variables: result_df, total_series_count

lines = []
lines.append(f"Scanned total distinct component-KPI series: {int(total_series_count)}.")
lines.append("Found component-KPI series with values >= their global P95 inside the incident window (showing up to top 20 by exceed count):")
lines.append("")
# Header for readability
lines.append("Rank. cmdb_id | kpi_name | global_p95 | window_count_exceed_p95 | window_total_points | earliest_exceed_timestamp (unix) | window_max_value")
for i, row in enumerate(result_df.itertuples(index=False), start=1):
    lines.append(
        f"{i}. {row.cmdb_id} | {row.kpi_name} | {float(row.global_p95):g} | "
        f"{int(row.window_count_exceed_p95)} | {int(row.window_total_points)} | "
        f"{int(row.window_earliest_exceed_timestamp) if not pd.isna(row.window_earliest_exceed_timestamp) else 'null'} | {float(row.window_max_value):g}"
    )

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id kpi_name   global_p95  window_total_points  window_count_exceed_p95  window_earliest_exceed_timestamp  window_max_value
0        ts-voucher-service      mem   38477824.0                   25                       25                        1705982280        38477824.0
1        ts-voucher-service   diskio          0.0                   25                       25                        1705982280               0.0
2          ts-voucher-mysql   socket          2.0                   25                       25                        1705982280               2.0
3          ts-voucher-mysql      mem  600387584.0                   25                       25                        1705982280       600387584.0
4          ts-voucher-mysql   diskio          0.0                   25                       25                        1705982280               0.0
5        ts-voucher-service   socket          4.0                   25                       25                        1705982280               4.0
6            ts-price-mongo   socket          5.0                   25                       25                        1705982280               5.0
7     ts-notification-mongo   socket          2.0                   25                       25                        1705982280               2.0
8         ts-avatar-service   diskio          0.0                   25                       25                        1705982280               0.0
9         ts-avatar-service   socket          5.0                   25                       25                        1705982280               5.0
10          ts-news-service   socket          1.0                   25                       25                        1705982280               1.0
11            ts-user-mongo   socket          4.0                   25                       25                        1705982280               4.0
12          ts-ui-dashboard   diskio          0.0                   25                       25                        1705982280               0.0
13  ts-inside-payment-mongo   socket          4.0                   25                       25                        1705982280               4.0
14  ts-admin-travel-service    error          0.0                   25                       25                        1705982280               0.0
15           ts-train-mongo   socket          6.0                   25                       25                        1705982280               6.0
16         ts-consign-mongo   socket          4.0                   25                       25                        1705982280               4.0
17            ts-food-mysql   socket          2.0                   25                       25                        1705982280               2.0
18   ts-consign-price-mongo   socket          4.0                   25                       25                        1705982280               4.0
19            ts-food-mysql      mem  599863296.0                   25                       25                        1705982280       599863296.0, 367)```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Candidate services
services = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

# 1) Compute global P95 per (cmdb_id, kpi_name) for the five services using full series (no window filtering)
# If a precomputed p95 exists in the environment, reuse it when possible.
if 'p95' in globals():
    p95_full = p95.copy()
else:
    p95_full = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'], observed=True)['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value': 'global_p95'})
    )

p95_five = p95_full[p95_full['cmdb_id'].isin(services)].reset_index(drop=True)
total_series_scanned = p95_five.shape[0]

# Build map for quick lookup
p95_map = {(r.cmdb_id, r.kpi_name): r.global_p95 for r in p95_five.itertuples(index=False)}

# 2) Prepare timestamps and filter to candidate services
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

df_candidates = df_metric[df_metric['cmdb_id'].isin(services)].copy()

# 3) Filter to incident window (UTC)
window_start = pd.to_datetime("2024-01-23 03:55:03", utc=True)
window_end   = pd.to_datetime("2024-01-23 04:25:03", utc=True)
df_window = df_candidates[(df_candidates['timestamp_dt'] >= window_start) & (df_candidates['timestamp_dt'] <= window_end)].copy()

# 4) For each (cmdb_id, kpi_name) detect consecutive runs where value >= global_p95
runs = []
group_cols = ['cmdb_id', 'kpi_name']
for (cmdb_id, kpi_name), grp in df_window.groupby(group_cols, observed=True):
    key = (cmdb_id, kpi_name)
    if key not in p95_map:
        # No global p95 computed for this series (skip)
        continue
    global_p95 = p95_map[key]
    g = grp.sort_values('timestamp').copy()
    g['exceed'] = g['value'] >= global_p95
    if g['exceed'].any():
        # Identify consecutive blocks of identical exceed boolean
        g['block'] = (g['exceed'] != g['exceed'].shift(1)).cumsum()
        # Aggregate only blocks where exceed == True
        exceed_blocks = g[g['exceed']].groupby('block', observed=True).agg(
            run_start_timestamp = ('timestamp', 'min'),
            run_end_timestamp   = ('timestamp', 'max'),
            run_length          = ('timestamp', 'size'),
            run_max_value       = ('value', 'max')
        ).reset_index(drop=True)
        # Keep runs with length >= 3
        exceed_blocks = exceed_blocks[exceed_blocks['run_length'] >= 3]
        for _, r in exceed_blocks.iterrows():
            runs.append({
                'cmdb_id': cmdb_id,
                'kpi_name': kpi_name,
                'global_p95': global_p95,
                'run_start_timestamp': int(r['run_start_timestamp']),
                'run_end_timestamp': int(r['run_end_timestamp']),
                'run_length': int(r['run_length']),
                'run_max_value': float(r['run_max_value'])
            })

# 5) Build result DataFrame
if runs:
    result_runs_df = pd.DataFrame(runs)
    result_runs_df = result_runs_df.sort_values(by=['run_length', 'run_start_timestamp'], ascending=[False, True]).reset_index(drop=True)
    result_runs_df = result_runs_df.head(50)
else:
    # empty dataframe with required columns
    result_runs_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_p95','run_start_timestamp','run_end_timestamp','run_length','run_max_value'
    ])

total_runs_found = result_runs_df.shape[0]

# Display compact outputs: result_runs_df (up to 50 rows) and summary info
result_runs_df, total_series_scanned, total_runs_found

```
Out[3]:
```
```python
# Build a plain-English summary string from existing results (reusing variables)
import pandas as pd

# Use existing variables: result_runs_df, total_series_scanned, total_runs_found
rows = []
rows.append(f"Scanned {int(total_series_scanned)} distinct (cmdb_id, kpi_name) series across the five candidate services.")
rows.append(f"Detected {int(total_runs_found)} anomaly runs (consecutive points >= global P95 with length >= 3) in the incident window.")
rows.append("")
if total_runs_found > 0:
    rows.append("Anomaly runs (top by length):")
    for i, r in enumerate(result_runs_df.itertuples(index=False), start=1):
        start_iso = pd.to_datetime(int(r.run_start_timestamp), unit='s', utc=True).isoformat()
        end_iso = pd.to_datetime(int(r.run_end_timestamp), unit='s', utc=True).isoformat()
        rows.append(
            f"{i}. {r.cmdb_id} | {r.kpi_name} | global_p95={r.global_p95:g} | "
            f"start={r.run_start_timestamp} ({start_iso}) | end={r.run_end_timestamp} ({end_iso}) | "
            f"length={int(r.run_length)} | max_value={r.run_max_value:g}"
        )
summary = "\n".join(rows)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id kpi_name  global_p95  run_start_timestamp  run_end_timestamp  run_length  run_max_value
0   ts-auth-service    error         0.0           1705982280         1705983720          25            0.0
1  ts-order-service    error         0.0           1705982280         1705983720          25            0.0, 37, 2)```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse df_trace if available, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure timestamp is integer and add UTC datetime column
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_trace['timestamp_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Service substrings to match
services = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

# Identify trace_name values that contain any of the substrings
pattern = "|".join(services)
matching_mask = df_trace['trace_name'].str.contains(pattern, na=False)
matching_trace_names = sorted(df_trace.loc[matching_mask, 'trace_name'].unique())

# Compute global P95 per trace_name using the full series (no time filtering)
p95_all = (
    df_trace
    .groupby('trace_name', observed=True)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)

# Keep only the trace_names that matched the service substrings
p95_sel = p95_all[p95_all['trace_name'].isin(matching_trace_names)].reset_index(drop=True)
total_series_scanned = p95_sel.shape[0]

# Merge global_p95 back into full trace dataframe
df_trace_p = df_trace.merge(p95_all, on='trace_name', how='left')

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-23 03:55:03", utc=True)
window_end   = pd.to_datetime("2024-01-23 04:25:03", utc=True)

# Filter to matching trace_names and incident window (inclusive)
df_window = df_trace_p[
    (df_trace_p['trace_name'].isin(matching_trace_names)) &
    (df_trace_p['timestamp_dt'] >= window_start) &
    (df_trace_p['timestamp_dt'] <= window_end)
].copy()

# Determine exceedance using global_p95 (computed from full series)
df_window['exceed'] = df_window['value'] >= df_window['global_p95']

# Aggregate per trace_name
agg = (
    df_window
    .groupby(['trace_name', 'global_p95'], observed=True)
    .agg(
        window_total_points = ('value', 'size'),
        window_count_exceed_p95 = ('exceed', 'sum'),
        window_max_value = ('value', 'max')
    )
    .reset_index()
)

# Earliest exceed timestamp per trace_name (raw unix seconds)
earliest_exceed = (
    df_window[df_window['exceed']]
    .groupby('trace_name', observed=True)['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp': 'window_earliest_exceed_timestamp'})
)

# Merge earliest exceed timestamp
result = agg.merge(earliest_exceed, on='trace_name', how='left')

# Keep only rows with at least one exceedance
result = result[result['window_count_exceed_p95'] > 0].copy()

# Sort by window_count_exceed_p95 descending and limit to top 20
result = result.sort_values(by='window_count_exceed_p95', ascending=False).head(20)

# Final compact result columns and reset index
result_traces_df = result[
    ['trace_name', 'global_p95', 'window_total_points', 'window_count_exceed_p95',
     'window_earliest_exceed_timestamp', 'window_max_value']
].reset_index(drop=True)

# Display compact outputs
result_traces_df, total_series_scanned

```
Out[4]:
```
```python
# Build a plain-English summary from previously computed result_traces_df and total_series_scanned
import pandas as pd

# Reuse existing variables: result_traces_df, total_series_scanned
total_scanned = int(total_series_scanned)
rows = []
rows.append(f"Scanned {total_scanned} distinct trace_name series that reference the five candidate services.")
total_found = int(result_traces_df.shape[0])
rows.append(f"Found {total_found} trace_name series with values >= their global P95 inside the incident window (2024-01-23 03:55:03 UTC to 2024-01-23 04:25:03 UTC).")
rows.append("")
rows.append("Top trace_name entries (trace_name | global_p95 | window_count_exceed_p95 | window_total_points | earliest_exceed_ts (unix / ISO) | window_max_value):")
for r in result_traces_df.itertuples(index=False):
    earliest_ts = r.window_earliest_exceed_timestamp
    if pd.isna(earliest_ts):
        earliest_display = "null"
    else:
        iso = pd.to_datetime(int(earliest_ts), unit='s', utc=True).isoformat()
        earliest_display = f"{int(earliest_ts)} / {iso}"
    rows.append(
        f"- {r.trace_name} | {float(r.global_p95):g} | {int(r.window_count_exceed_p95)} | {int(r.window_total_points)} | {earliest_display} | {float(r.window_max_value):g}"
    )

summary = "\n".join(rows)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                    trace_name   global_p95  window_total_points  window_count_exceed_p95  window_earliest_exceed_timestamp  window_max_value
0         trace.to_ts-order-service.error_rate     0.000000                  170                      170                        1705982280          0.000000
1      trace.from_ts-travel-service.error_rate     0.000000                  150                      150                        1705982280          0.000000
2        trace.to_ts-travel-service.error_rate     0.000000                  147                      147                        1705982280          0.000000
3         trace.to_ts-train-service.error_rate     0.000000                  100                      100                        1705982280          0.000000
4         trace.to_ts-route-service.error_rate     0.000000                  100                      100                        1705982280          0.000000
5          trace.to_ts-auth-service.error_rate     0.000000                   50                       50                        1705982280          0.000000
6       trace.from_ts-order-service.error_rate     0.000000                   49                       49                        1705982280          0.000000
7        trace.from_ts-auth-service.error_rate     0.000000                   25                       25                        1705982280          0.000000
8       trace.from_ts-route-service.error_rate     0.000000                   25                       25                        1705982280          0.000000
9       trace.from_ts-train-service.error_rate     0.000000                   25                       25                        1705982280          0.000000
10     trace.to_ts-order-service.duration_mean     0.665049                  170                        9                        1705982880          0.735786
11      trace.to_ts-order-service.duration_p95     0.691016                  170                        9                        1705982880          0.801781
12         trace.to_ts-order-service.row_count   463.300000                  170                        9                        1705982280        572.000000
13   trace.from_ts-travel-service.duration_p95     0.060975                  150                        8                        1705982580          0.170516
14  trace.from_ts-travel-service.duration_mean     0.047603                  150                        8                        1705982580          0.072034
15    trace.to_ts-travel-service.duration_mean     0.604241                  147                        8                        1705982580          0.772370
16     trace.to_ts-travel-service.duration_p95     0.810131                  147                        8                        1705982580          1.651787
17        trace.to_ts-travel-service.row_count  1608.700000                  147                        8                        1705982280       1816.000000
18      trace.from_ts-travel-service.row_count  1605.550000                  150                        8                        1705982280       1816.000000
19         trace.to_ts-train-service.row_count  2624.000000                  100                        6                        1705982700       2708.000000, 40)```
```

In [ ]:
# In[5]:

import pandas as pd

# Reuse existing df_metric if available, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure timestamp columns exist and correct types
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
if 'timestamp_dt' not in df_metric.columns:
    df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Target component
component = 'ts-travel-service'

# 1) Compute global P95 per kpi_name for the component using the full series (no time filtering)
df_travel_full = df_metric[df_metric['cmdb_id'] == component].copy()
p95_travel = (
    df_travel_full
    .groupby('kpi_name', observed=True)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)

# Count distinct kpi_name series scanned
total_kpi_series = p95_travel.shape[0]

# 2) Merge global_p95 back into the travel series
df_travel_full = df_travel_full.merge(p95_travel, on='kpi_name', how='left')

# 3) Filter to incident window (UTC)
window_start = pd.to_datetime("2024-01-23 03:55:03", utc=True)
window_end   = pd.to_datetime("2024-01-23 04:25:03", utc=True)
df_travel_window = df_travel_full[(df_travel_full['timestamp_dt'] >= window_start) & (df_travel_full['timestamp_dt'] <= window_end)].copy()

# 4) Determine exceedance using global_p95 (computed from full series)
df_travel_window['exceed'] = df_travel_window['value'] >= df_travel_window['global_p95']

# 5) Aggregate per kpi_name
agg = (
    df_travel_window
    .groupby(['kpi_name', 'global_p95'], observed=True)
    .agg(
        window_total_points = ('value', 'size'),
        window_count_exceed_p95 = ('exceed', 'sum'),
        window_max_value = ('value', 'max')
    )
    .reset_index()
)

# 6) Earliest exceed timestamp per kpi_name (raw unix seconds)
earliest_exceed = (
    df_travel_window[df_travel_window['exceed']]
    .groupby('kpi_name', observed=True)['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp': 'window_earliest_exceed_timestamp'})
)

# 7) Merge earliest exceed timestamp
result = agg.merge(earliest_exceed, on='kpi_name', how='left')

# 8) Keep only rows with at least one exceedance, sort and limit to top 20
result = result[result['window_count_exceed_p95'] > 0].copy()
result = result.sort_values(by='window_count_exceed_p95', ascending=False).head(20)

# 9) Select and present compact columns
result_ts_travel = result[
    ['kpi_name', 'global_p95', 'window_total_points', 'window_count_exceed_p95',
     'window_earliest_exceed_timestamp', 'window_max_value']
].reset_index(drop=True)

# Display compact result and total number of distinct kpi_name series scanned
result_ts_travel, total_kpi_series

```
Out[5]:
```
Scanned 7 distinct kpi_name series for cmdb_id = ts-travel-service.

Found the following kpi_name entries with values >= their global P95 inside the incident window (2024-01-23 03:55:03 UTC to 2024-01-23 04:25:03 UTC):

- cpu — global_p95 = 3.492234; window_total_points = 25; window_count_exceed_p95 = 2; earliest_exceed_timestamp = 1705982280 (2024-01-23T03:58:00Z); window_max_value = 3.571213
- diskio — global_p95 = 57736.7; window_total_points = 25; window_count_exceed_p95 = 2; earliest_exceed_timestamp = 1705982460 (2024-01-23T04:01:00Z); window_max_value = 60645.43
- latency-50 — global_p95 = 0.0186966; window_total_points = 25; window_count_exceed_p95 = 2; earliest_exceed_timestamp = 1705982820 (2024-01-23T04:07:00Z); window_max_value = 0.01870677
- latency-90 — global_p95 = 0.2615143; window_total_points = 25; window_count_exceed_p95 = 2; earliest_exceed_timestamp = 1705983000 (2024-01-23T04:09:00Z); window_max_value = 0.3160975
- mem — global_p95 = 269,573,400 (≈2.695734e+08); window_total_points = 25; window_count_exceed_p95 = 2; earliest_exceed_timestamp = 1705983600 (2024-01-23T04:20:00Z); window_max_value ≈ 269,580,300
- socket — global_p95 = 17.46333; window_total_points = 25; window_count_exceed_p95 = 2; earliest_exceed_timestamp = 1705982280 (2024-01-23T03:58:00Z); window_max_value = 17.61404
- workload — global_p95 = 10.25544; window_total_points = 25; window_count_exceed_p95 = 2; earliest_exceed_timestamp = 1705983600 (2024-01-23T04:20:00Z); window_max_value = 13.9665

All reported kpis had exactly 2 points in the incident window meeting or exceeding their global P95.

The original code execution output of IPython Kernel is also provided below for reference:

(     kpi_name    global_p95  window_total_points  window_count_exceed_p95  window_earliest_exceed_timestamp  window_max_value
0         cpu  3.492234e+00                   25                        2                        1705982280      3.571213e+00
1      diskio  5.773670e+04                   25                        2                        1705982460      6.064543e+04
2  latency-50  1.869660e-02                   25                        2                        1705982820      1.870677e-02
3  latency-90  2.615143e-01                   25                        2                        1705983000      3.160975e-01
4         mem  2.695734e+08                   25                        2                        1705983600      2.695803e+08
5      socket  1.746333e+01                   25                        2                        1705982280      1.761404e+01
6    workload  1.025544e+01                   25                        2                        1705983600      1.396650e+01, 7)```
```

In [ ]:
# In[6]:

import pandas as pd

# Reuse existing df_log if present; otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")

# Ensure timestamp is integer and add UTC datetime column
df_log['timestamp'] = df_log['timestamp'].astype(int)
df_log['timestamp_dt'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Filter for the component and incident window (inclusive)
component = 'ts-travel-service'
window_start = pd.to_datetime("2024-01-23 03:55:03", utc=True)
window_end   = pd.to_datetime("2024-01-23 04:25:03", utc=True)

df_filtered = df_log[
    (df_log['cmdb_id'] == component) &
    (df_log['timestamp_dt'] >= window_start) &
    (df_log['timestamp_dt'] <= window_end)
].copy()

# Prepare outputs
if df_filtered.empty:
    counts_by_log = pd.DataFrame(columns=['log_name', 'count'])
    sample_rows = pd.DataFrame(columns=['timestamp', 'cmdb_id', 'log_name', 'value'])
    empty_result = "EMPTY: no log rows for ts-travel-service in the incident window"
else:
    counts_by_log = (
        df_filtered
        .groupby('log_name', observed=True)['value']
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .reset_index(drop=True)
    )
    sample_rows = (
        df_filtered[['timestamp', 'cmdb_id', 'log_name', 'value']]
        .sort_values('timestamp', ascending=True)
        .head(50)
        .reset_index(drop=True)
    )
    empty_result = None

# Return compact outputs: 1) counts per log_name, 2) up to 50 log rows, 3) empty indicator (None if not empty)
counts_by_log, sample_rows, empty_result

```
Out[6]:
```
Summary for ts-travel-service logs in the incident window (2024-01-23 03:55:03 UTC to 2024-01-23 04:25:03 UTC):

- Total rows found: 50.
- Time range in the results: earliest timestamp 1705982280 (2024-01-23T03:58:00Z), latest timestamp 1705983720 (2024-01-23T04:22:00Z).
- Counts by log_name:
  - log.error_count: 25
  - log.row_count: 25
- Observations:
  - All log.error_count entries in the window are 0 (no recorded errors).
  - log.row_count values vary; in the returned rows they range roughly from 70 up to about 2001 (most values are in the ~1700–2000 range, with a low of 70 at timestamp 1705983720).
- If you want, I can display the full 50 log rows (timestamp, cmdb_id, log_name, value) or filter them further.

The original code execution output of IPython Kernel is also provided below for reference:

(          log_name  count
0  log.error_count     25
1    log.row_count     25,      timestamp            cmdb_id         log_name  value
0   1705982280  ts-travel-service  log.error_count      0
1   1705982280  ts-travel-service    log.row_count   2001
2   1705982340  ts-travel-service  log.error_count      0
3   1705982340  ts-travel-service    log.row_count   1711
4   1705982400  ts-travel-service  log.error_count      0
..         ...                ...              ...    ...
45  1705983600  ts-travel-service    log.row_count   1995
46  1705983660  ts-travel-service  log.error_count      0
47  1705983660  ts-travel-service    log.row_count   1772
48  1705983720  ts-travel-service  log.error_count      0
49  1705983720  ts-travel-service    log.row_count     70

[50 rows x 4 columns], None)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```